# Communicating with Plots: Labels, Annotations, and Scales

**How many plots did you throw away on your last project?**

In exploratory data analysis you make dozens of plots, glance at each one, and move on.
Almost none of those figures are meant for anyone else.

When you *communicate* results, the audience does not share your context.
They need titles that state the finding, labels with units, and scales that match how people read numbers.
This notebook covers three levers: **labels**, **annotations**, and **scales**.

**Learning objectives**

- Add titles, axis labels, subtitles, and captions with matplotlib and seaborn.
- Annotate groups and outliers on a plot.
- Adjust tick breaks, formatters, color palettes, log scales, and zoom limits.


## Prerequisites

- Python, **pandas**, **matplotlib**, and **seaborn**.
- Datasets from the original ggplot2 package, fetched from [Rdatasets](https://vincentarelbundock.github.io/Rdatasets/).
- Prior EDA notebooks in this course (variation and covariation).


In [ ]:
MPG_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/mpg.csv"
DIAMONDS_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/diamonds.csv"
PRESIDENTIAL_URL = "https://vincentarelbundock.github.io/Rdatasets/csv/ggplot2/presidential.csv"

import textwrap

import numpy as np  # https://numpy.org/doc/
import pandas as pd  # https://pandas.pydata.org/docs/
import matplotlib.pyplot as plt  # https://matplotlib.org/stable/
import seaborn as sns  # https://seaborn.pydata.org/
from matplotlib import dates as mdates
from matplotlib import ticker

sns.set_theme()

mpg = pd.read_csv(MPG_URL, index_col=0)
diamonds = pd.read_csv(DIAMONDS_URL, index_col=0)
presidential = pd.read_csv(PRESIDENTIAL_URL, index_col=0, parse_dates=["start", "end"])


## Labels

The easiest upgrade from an exploratory graphic to an expository one is **good labels**.

- Replace raw column names with readable text and units.
- Use the **title** to summarize the main finding, not to describe the chart type.
- Use a **subtitle** for extra detail and a **caption** for the data source.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=mpg,
    x="displ",  # engine displacement in litres
    y="hwy",  # highway fuel economy in mpg
    hue="class",  # vehicle class for color
    ax=ax,
)
sns.regplot(
    data=mpg,
    x="displ",
    y="hwy",
    scatter=False,
    lowess=True,  # smooth trend without a parametric assumption
    color="black",
    ax=ax,
)

ax.set_xlabel("Engine displacement (L)")
ax.set_ylabel("Highway fuel economy (mpg)")
ax.set_title("Fuel efficiency generally decreases with engine size")
fig.suptitle(
    "Two seaters (sports cars) are an exception because of their light weight",
    fontsize=10,
    y=0.98,
)
handles, labels = ax.get_legend_handles_labels()
ax.legend(handles=handles, labels=labels, title="Car type")
fig.text(0.99, 0.01, "Data from fueleconomy.gov", ha="right", fontsize=9)
plt.tight_layout()
plt.show()


Avoid titles like *"Scatterplot of engine displacement vs. fuel economy"* — they repeat what the axes already show.

You can put **mathematical notation** on axis labels using matplotlib's mathtext:


In [ ]:
df = pd.DataFrame({"x": np.arange(1, 11), "y": np.cumsum(np.arange(1, 11) ** 2)})

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(data=df, x="x", y="y", ax=ax)
ax.set_xlabel(r"$x_i$")
ax.set_ylabel(r"$\sum_{i=1}^{n} x_i^2$")
plt.show()


## Annotations

Labels describe the whole plot; **annotations** call out specific observations or groups.

- Build a small table of points to label (for example, the largest engine in each drive type).
- Place text with `ax.annotate` or `ax.text`.
- Use reference lines (`ax.axhline`, `ax.axvline`) or arrows (`annotate(..., arrowprops=...)`) for trends.


In [ ]:
drive_labels = {"f": "front-wheel drive", "r": "rear-wheel drive", "4": "4-wheel drive"}

label_info = mpg.sort_values("displ", ascending=False).groupby("drv").head(1).copy()
label_info["drive_type"] = label_info["drv"].map(drive_labels)
label_info


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.scatterplot(
    data=mpg,
    x="displ",
    y="hwy",
    hue="drv",
    alpha=0.3,
    legend=False,
    ax=ax,
)
for drv, group in mpg.groupby("drv"):
    sns.regplot(
        data=group,
        x="displ",
        y="hwy",
        scatter=False,
        lowess=True,
        ax=ax,
        label=drv,
    )

for _, row in label_info.iterrows():
    ax.annotate(
        row["drive_type"],
        xy=(row["displ"], row["hwy"]),
        xytext=(8, 8),
        textcoords="offset points",
        fontsize=11,
        fontweight="bold",
    )

ax.set_xlabel("Engine displacement (L)")
ax.set_ylabel("Highway fuel economy (mpg)")
plt.show()


To **highlight outliers**, filter a subset and label it — here, unusually efficient cars or large engines with decent mileage:


In [ ]:
potential_outliers = mpg[(mpg["hwy"] > 40) | ((mpg["hwy"] > 20) & (mpg["displ"] > 5))]

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", ax=ax)
sns.scatterplot(
    data=potential_outliers,
    x="displ",
    y="hwy",
    color="red",
    s=120,
    linewidth=2,
    edgecolor="red",
    facecolors="none",
    ax=ax,
)

for _, row in potential_outliers.iterrows():
    ax.annotate(
        row["model"],
        xy=(row["displ"], row["hwy"]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

ax.set_xlabel("Engine displacement (L)")
ax.set_ylabel("Highway fuel economy (mpg)")
plt.show()


In [ ]:
trend_text = textwrap.fill(
    "Larger engine sizes tend to have lower fuel economy.",
    width=30,
)
trend_text


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", ax=ax)

ax.annotate(
    trend_text,
    xy=(3.5, 38),
    xytext=(3.5, 38),
    color="red",
    fontsize=10,
    ha="left",
    bbox=dict(boxstyle="round", facecolor="white", edgecolor="red"),
)
ax.annotate(
    "",
    xy=(5, 25),
    xytext=(3, 35),
    arrowprops=dict(arrowstyle="->", color="red", lw=2),
)

ax.set_xlabel("Engine displacement (L)")
ax.set_ylabel("Highway fuel economy (mpg)")
plt.show()


## Scales

**Scales** control how values map to positions, colors, and legend entries.

- Matplotlib and seaborn choose sensible defaults (tick positions, color palettes).
- You override them when you know more about the data or the audience.


### Axis ticks and legend keys

Axes and legends are called **guides**.

- **Breaks** — where ticks sit.
- **Labels** — text at those ticks (or legend entries).


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", hue="drv", ax=ax)
ax.set_yticks(np.arange(15, 41, 5))
plt.show()


In [ ]:
drv_names = {"4": "4-wheel", "f": "front", "r": "rear"}

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", hue="drv", ax=ax)
ax.set_xticklabels([])
ax.set_yticklabels([])
handles, labels = ax.get_legend_handles_labels()
ax.legend(
    handles=handles,
    labels=[drv_names.get(l, l) for l in labels],
    title="Drive train",
)
plt.show()


Format numbers as **currency** or **percent** with tick formatters:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

sns.boxplot(data=diamonds, x="price", y="cut", orient="h", ax=axes[0])
axes[0].xaxis.set_major_formatter(ticker.StrMethodFormatter("${x:,.0f}"))

sns.boxplot(data=diamonds, x="price", y="cut", orient="h", ax=axes[1])
axes[1].set_xticks(np.arange(1000, 20000, 6000))
axes[1].xaxis.set_major_formatter(
    ticker.FuncFormatter(lambda v, _pos: f"${v / 1000:,.0f}K")
)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(
    data=diamonds,
    x="cut",
    hue="clarity",
    multiple="fill",
    shrink=0.8,
    ax=ax,
)
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1.0))
ax.set_ylabel("Percentage")
plt.show()


For **sparse** data, set breaks to the actual observation values — here, presidential terms:


In [ ]:
pres = presidential.copy()
pres["id"] = 33 + np.arange(1, len(pres) + 1)

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in pres.iterrows():
    ax.plot(
        [row["start"], row["end"]],
        [row["id"], row["id"]],
        color="gray",
        linewidth=2,
    )
sns.scatterplot(data=pres, x="start", y="id", ax=ax)
ax.set_xticks(pres["start"])
ax.xaxis.set_major_formatter(mdates.DateFormatter("'%y"))
ax.set_xlabel(None)
plt.show()


### Legend layout

Move the legend with `ax.legend(loc=...)` or `bbox_to_anchor`.

- Wide plots: legend on top or bottom.
- Tall plots: legend on the left or right.
- `legend=False` when direct labels replace the legend.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
positions = ["right", "center left", "upper center", "lower center"]

for ax, loc in zip(axes.flat, positions):
    sns.scatterplot(data=mpg, x="displ", y="hwy", hue="class", ax=ax)
    ax.legend(loc=loc, ncol=3 if loc in ("upper center", "lower center") else 1)
    ax.set_title(f"legend loc={loc!r}")

plt.tight_layout()
plt.show()


When points use a low `alpha`, the legend markers can be hard to read.

- Enlarge them with `markerscale` (matplotlib's equivalent of ggplot2's `override.aes`).
- Spread a long legend over several columns with `ncol`, and move it below the plot.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=mpg, x="displ", y="hwy", hue="class", alpha=0.3, ax=ax)
sns.regplot(data=mpg, x="displ", y="hwy", scatter=False, lowess=True, color="black", ax=ax)
ax.legend(
    title="Car type",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.15),  # push the legend below the axes
    ncol=4,  # lay the entries out in four columns
    markerscale=2,  # draw legend markers twice the size of the plotted points
)
plt.show()

### Replacing a scale

**Log scales** make skewed relationships easier to read while keeping axis labels on the original units:


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.histplot(data=diamonds, x="carat", y="price", cbar=True, ax=axes[0])
axes[0].set_title("Raw scale")

sns.histplot(data=diamonds, x="carat", y="price", cbar=True, ax=axes[1])
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_title("Log axes (original tick labels)")

plt.tight_layout()
plt.show()


Choose a **color palette** for accessibility — seaborn's `palette` argument accepts named palettes such as 'Set1' or 'viridis':


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.scatterplot(data=mpg, x="displ", y="hwy", hue="drv", ax=axes[0])
axes[0].set_title("Default palette")

sns.scatterplot(
    data=mpg,
    x="displ",
    y="hwy",
    hue="drv",
    style="drv",
    palette="Set1",
    ax=axes[1],
)
axes[1].set_title("Set1 + redundant shape")

plt.tight_layout()
plt.show()


In [ ]:
party_colors = {"Republican": "#E81B23", "Democratic": "#00AEF3"}

pres = presidential.copy()
pres["id"] = 33 + np.arange(1, len(pres) + 1)

fig, ax = plt.subplots(figsize=(10, 6))
for _, row in pres.iterrows():
    color = party_colors[row["party"]]
    ax.plot([row["start"], row["end"]], [row["id"], row["id"]], color=color, lw=2)
sns.scatterplot(data=pres, x="start", y="id", hue="party", palette=party_colors, ax=ax)
plt.show()


In [ ]:
rng = np.random.default_rng(0)
df = pd.DataFrame({"x": rng.normal(size=5000), "y": rng.normal(size=5000)})

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
configs = [
    ("Default", None, 30),
    ("Viridis", "viridis", 30),
    ("Viridis (fewer bins)", "viridis", 12),
]

for ax, (title, cmap, bins) in zip(axes, configs):
    kwargs = {"cmap": cmap} if cmap else {}
    sns.histplot(
        data=df,
        x="x",
        y="y",
        bins=bins,
        cbar=True,
        ax=ax,
        **kwargs,
    )
    ax.set_title(title)
    ax.set_aspect("equal")

plt.tight_layout()
plt.show()


### Zooming

Three ways to control what appears in the plotting window:

1. **Filter the data** before plotting — scales and smoothers refit to the subset.
2. **Set axis limits** — same as filtering for the view, but can hide data.
3. **Set limits after plotting** (`set_xlim` / `set_ylim`) — zoom without refitting smoothers.


In [ ]:
subset = mpg[(mpg["displ"] >= 5) & (mpg["displ"] <= 6) & (mpg["hwy"] >= 10) & (mpg["hwy"] <= 25)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data in zip(axes, [mpg, subset]):
    sns.scatterplot(data=data, x="displ", y="hwy", hue="drv", ax=ax)
    sns.regplot(data=data, x="displ", y="hwy", scatter=False, lowess=True, color="black", ax=ax)

axes[0].set_title("All data")
axes[1].set_title("Filtered data")
plt.tight_layout()
plt.show()


In [ ]:
subset = mpg[(mpg["displ"] >= 5) & (mpg["displ"] <= 6) & (mpg["hwy"] >= 10) & (mpg["hwy"] <= 25)]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sns.scatterplot(data=subset, x="displ", y="hwy", hue="drv", ax=axes[0])
sns.regplot(data=subset, x="displ", y="hwy", scatter=False, lowess=True, color="black", ax=axes[0])
axes[0].set_title("Data filtered, then smoothed")

sns.scatterplot(data=mpg, x="displ", y="hwy", hue="drv", ax=axes[1])
sns.regplot(data=mpg, x="displ", y="hwy", scatter=False, lowess=True, color="black", ax=axes[1])
axes[1].set_xlim(5, 6)
axes[1].set_ylim(10, 25)
axes[1].set_title("Full data smoothed, then zoomed")

plt.tight_layout()
plt.show()


In [ ]:
suv = mpg[mpg["class"] == "suv"]
compact = mpg[mpg["class"] == "compact"]
xlim = mpg["displ"].min(), mpg["displ"].max()
ylim = mpg["hwy"].min(), mpg["hwy"].max()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, data, title in zip(axes, [suv, compact], ["SUV", "Compact"]):
    sns.scatterplot(data=data, x="displ", y="hwy", hue="drv", ax=ax)
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_title(title)

plt.tight_layout()
plt.show()
